# Complete Unsloth Colab — AIRIS Path A (recommended) + optional legacy tool-string format

**Runtime:** Runtime → Change runtime type → **T4 GPU**.

## Two dataset modes (auto-detected from the first JSONL record)

1. **`airis_path_a`** (recommended for Cursor / AIRIS): each line is an object with `schema_version`, `system`, `history`, `user`, `assistant_raw`.
   - Training uses Hugging Face **chat messages** + `tokenizer.apply_chat_template` (same family as production OpenAI-compatible chat).
   - `assistant_raw` may contain `<<<EXECUTION` … `>>>END` blocks — **not** OpenAI `function_call` / `<tool_call>` JSON.

2. **`legacy_messages`** (non-AIRIS teacher runs): each line is `{"messages": [...]}` with optional `function_call` / `tool` roles.
   - Set `USE_LEGACY_LLAMA_TOOL_STRING = True` in the config cell to use the Llama-style `<tool_call>` string converter (only if you intentionally train that I/O).

## Before Colab

- For AIRIS data, validate in the repo: `npm run sft:validate-path-a -- your.jsonl`
- Upload JSONL to Drive: e.g. `MyDrive/agent_finetune/agent_training_data.jsonl`


### Cell 1 — Install (run first; `%%capture` hides long pip logs)


In [ ]:
%%capture
!pip install -q unsloth
!pip uninstall -y -q unsloth 2>/dev/null
!pip install -q --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes datasets huggingface_hub sentencepiece protobuf

import torch
print("torch", torch.__version__, "CUDA", torch.cuda.is_available())


In [ ]:
# @title Cell 2 — Mount Google Drive & load JSONL
from google.colab import drive
import json, os

drive.mount("/content/drive")

# Default path (matches your folder layout) — change if needed
DATA_PATH = "/content/drive/MyDrive/agent_finetune/agent_training_data.jsonl"

assert os.path.isfile(DATA_PATH), f"Missing {DATA_PATH} — upload JSONL to Drive or fix DATA_PATH"

raw_samples = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        raw_samples.append(json.loads(line))

print(f"Loaded {len(raw_samples)} JSONL records")
print("\n--- FIRST RECORD (truncated) ---")
print(json.dumps(raw_samples[0], indent=2)[:2500])


In [ ]:
# @title Cell 3 — Config (paths, model, training, dataset mode)
# --- Data format ---
# If first record has assistant_raw + schema_version → AIRIS Path A (recommended).
# If it has "messages" → legacy; set USE_LEGACY_LLAMA_TOOL_STRING=True only if you want <tool_call> string training.
USE_LEGACY_LLAMA_TOOL_STRING = False

# --- Model ---
MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 4096  # lower to 2048 if T4 OOM; your 8192 may OOM on 8B 4-bit + LoRA

# --- Training ---
BATCH_SIZE = 1
GRAD_ACCUM = 4
NUM_EPOCHS = 3
MAX_STEPS = -1  # set >0 to cap steps instead of epochs
LEARNING_RATE = 2e-4
WARMUP_STEPS = 10
LOGGING_STEPS = 5
EVAL_STEPS = 20
SAVE_STEPS = 50
SEED = 42

# --- Output ---
SAVE_PATH = "/content/drive/MyDrive/agent_finetune/lora_adapter"
OUTPUT_DIR = "/content/drive/MyDrive/agent_finetune/colab_train_out"  # checkpoints

# --- Hugging Face Hub (optional) ---
HF_REPO_ID = ""  # e.g. "username/my-airis-lora"
HF_PRIVATE = True


In [ ]:
# @title Cell 4 — Imports + optional HF login (Colab secret HF_TOKEN)
import os
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from trl import SFTTrainer, SFTConfig
from huggingface_hub import login

if os.environ.get("HF_TOKEN"):
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)


## Cell 5 — Legacy tool-string converter (only if `USE_LEGACY_LLAMA_TOOL_STRING`)

**Not used for AIRIS Path A.** AIRIS uses fenced `<<<EXECUTION` blocks inside normal assistant text, not `<tool_call>` JSON.

If your `agent_training_data.jsonl` was built for OpenAI-style tools and you still want Llama-style tool strings, set `USE_LEGACY_LLAMA_TOOL_STRING = True` and **replace `TOOLS`** with the same JSON schemas your teacher used.


In [ ]:
# @title Cell 5 — REPLACE TOOLS only for legacy Llama tool-string mode
TOOLS = []

def convert_to_chat_format(messages, tools):
    """Legacy: Llama 3.1-style string with <tool_call> blocks (NOT AIRIS)."""
    import json as _json
    text = "<|begin_of_text|>"
    system_msg = next((m for m in messages if m.get("role") == "system"), None)
    sys_content = system_msg["content"] if system_msg else "You are a helpful AI assistant."
    tools_json = _json.dumps(tools, indent=2)
    system_block = f"{sys_content}\n\nYou have access to the following functions. Use them if required:\n{tools_json}"
    text += f"<|start_header_id|>system<|end_header_id|>\n\n{system_block}<|eot_id|>"
    for msg in messages:
        if msg.get("role") == "system":
            continue
        role = msg.get("role")
        if role == "user":
            text += f"<|start_header_id|>user<|end_header_id|>\n\n{msg['content']}<|eot_id|>"
        elif role == "assistant":
            content = ""
            fc = msg.get("function_call")
            if fc:
                args = fc.get("arguments")
                if isinstance(args, str):
                    try:
                        args = _json.loads(args)
                    except Exception:
                        pass
                tool_call_obj = {"name": fc.get("name"), "arguments": args}
                content = f"<tool_call>\n{_json.dumps(tool_call_obj)}\n</tool_call>"
            elif msg.get("content"):
                content = msg["content"]
            if content:
                text += f"<|start_header_id|>assistant<|end_header_id|>\n\n{content}<|eot_id|>"
        elif role in ("tool", "function"):
            resp = msg.get("content", "")
            text += f"<|start_header_id|>tool<|end_header_id|>\n\n{resp}<|eot_id|>"
    return text


In [ ]:
# @title Cell 6 — Detect format & build `text` training column
def detect_format(sample: dict) -> str:
    if isinstance(sample, dict) and sample.get("schema_version") == 1 and "assistant_raw" in sample:
        return "airis_path_a"
    if isinstance(sample, dict) and "messages" in sample:
        return "legacy_messages"
    if isinstance(sample, list):
        return "legacy_messages_list"
    return "unknown"

DATASET_FORMAT = detect_format(raw_samples[0])
print("DATASET_FORMAT =", DATASET_FORMAT)

if DATASET_FORMAT == "unknown":
    raise ValueError("Unrecognized JSONL: need Path A row or {messages: [...]}")

if DATASET_FORMAT == "airis_path_a":
    def path_a_row_to_messages(row: dict) -> list[dict]:
        messages = [{"role": "system", "content": row["system"]}]
        for turn in row.get("history") or []:
            role = turn.get("role")
            if role not in ("user", "assistant"):
                continue
            messages.append({"role": role, "content": turn.get("content", "")})
        messages.append({"role": "user", "content": row.get("user", "")})
        messages.append({"role": "assistant", "content": row.get("assistant_raw", "")})
        return messages

    # Defer apply_chat_template to after tokenizer is patched (next cell)
    path_a_messages = [path_a_row_to_messages(r) for r in raw_samples]
    formatted_data = None
elif DATASET_FORMAT in ("legacy_messages", "legacy_messages_list"):
    if not USE_LEGACY_LLAMA_TOOL_STRING:
        raise ValueError("Legacy messages detected: set USE_LEGACY_LLAMA_TOOL_STRING=True and fill TOOLS, or convert data to AIRIS Path A JSONL.")
    formatted_data = []
    for sample in raw_samples:
        messages = sample["messages"] if isinstance(sample, dict) else sample
        formatted_data.append({"text": convert_to_chat_format(messages, TOOLS)})
    path_a_messages = None
else:
    path_a_messages = None
    formatted_data = None

print("Prepared for training pipeline (tokenizer applied in next cell).")


In [ ]:
# @title Cell 7 — Load model + tokenizer + LoRA + chat template
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

model.print_trainable_parameters()

# Build final HF dataset with `text` (do NOT pre-tokenize; SFTTrainer tokenizes from `text`)
if DATASET_FORMAT == "airis_path_a":
    texts = [tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False) for conv in path_a_messages]
    hf_dataset = Dataset.from_list([{"text": t} for t in texts])
else:
    hf_dataset = Dataset.from_list(formatted_data)

split = hf_dataset.train_test_split(test_size=0.1, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]
print(train_ds, eval_ds)
print("\n--- text preview ---\n", train_ds[0]["text"][:1200])


In [ ]:
# @title Cell 8 — SFTTrainer (correct: raw `text`, no manual tokenize map)
train_kwargs = dict(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=WARMUP_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=LOGGING_STEPS,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
)
if MAX_STEPS and MAX_STEPS > 0:
    train_kwargs["max_steps"] = MAX_STEPS
else:
    train_kwargs["num_train_epochs"] = NUM_EPOCHS

args = SFTConfig(**train_kwargs, dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, packing=False)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=args,
)

# Train loss on assistant segments (multi-turn safe for repeated headers)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>" + chr(10) * 2,
    response_part="<|start_header_id|>assistant<|end_header_id|>" + chr(10) * 2,
)

print("Starting training...")
trainer.train()
print("Training complete.")


In [ ]:
# @title Cell 9 — Save LoRA to Google Drive
import os
os.makedirs(SAVE_PATH, exist_ok=True)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("Saved LoRA to", SAVE_PATH)
# Optional merged 16-bit (large):
# model.save_pretrained_merged(SAVE_PATH + "_merged_16bit", tokenizer, save_method="merged_16bit")


In [ ]:
# @title Cell 10 — (Optional) Push to Hugging Face Hub
if HF_REPO_ID:
    tok = os.environ.get("HF_TOKEN")
    model.push_to_hub(HF_REPO_ID, token=tok, private=HF_PRIVATE)
    tokenizer.push_to_hub(HF_REPO_ID, token=tok, private=HF_PRIVATE)
    print("Pushed to https://huggingface.co/" + HF_REPO_ID)
else:
    print("Skipping Hub push (HF_REPO_ID empty). Use Colab secret HF_TOKEN for gated models + push.")


In [ ]:
# @title Cell 11 — Quick inference (Path A: one user turn + optional system)
FastLanguageModel.for_inference(model)

if DATASET_FORMAT == "airis_path_a":
    system_prompt = "You are the AIRIS workspace agent. Reply with prose and use <<<EXECUTION ... >>>END when needed."
    user_prompt = "Add a note widget titled Demo with body hello from Colab."
    conv = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    prompt = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
else:
    prompt = convert_to_chat_format(
        [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "Hello"}],
        TOOLS,
    )

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=False)[:4000])


## After training — point Cursor / AIRIS at your server

- **Ollama:** serve a merged or adapter-backed model; in AIRIS set the OpenAI-compatible base URL (e.g. `http://localhost:11434/v1`) and model id.
- **vLLM / LiteLLM:** same idea: OpenAI-compatible chat completions with your fine-tuned instruct model.

AIRIS sends **plain** chat messages (`system` + `user`/`assistant` strings). Path A fine-tuning matches that; legacy `<tool_call>` training does **not** match AIRIS execution fences unless you add a separate translation layer at inference.
